# Practical SystemVerilog: From Digital Logic to Real Silicon
### A Survival Guide for Lab 2 on the Pico Dev-iCE

---

## 1. Hardware Description vs. Writing Software

If you have programmed in Python or C, your mental model of code is **sequential**: Line 1 runs, then Line 2 runs, then Line 3 runs.

In an FPGA, **there is no processor running your code**. When you write SystemVerilog, you are not writing instructions for a CPU—**you are drawing a schematic diagram using text**.

From your Digital Logic course, you know that all digital systems are composed of two distinct classes of circuits:

1. **Combinational Logic:** Networks of Boolean gates (AND, OR, NOT) and multiplexers. They have no memory. Their outputs react continuously to their inputs after a small propagation delay ($t_{pd}$).
2. **Sequential Logic:** Registers built from D Flip-Flops. They sample their inputs only on the active edge of a clock signal (satisfying setup time $t_{setup}$ and hold time $t_{hold}$).

```
                     +---------------------------------------+
                     |  Pico Dev-iCE FPGA (iCE40UP5K)        |
      clk ---------> |  [Sequential: D Flip-Flops]           |
      reset_n -----> |                 ^                     |
                     |                 | (State & Timers)    |
                     |                 v                     |
                     |  [Combinational: Logic Gates/Muxes]   | ---> led_red (Pin 39)
                     |                                       | ---> led_yel (Pin 40)
                     |                                       | ---> led_grn (Pin 41)
                     +---------------------------------------+
```

Every line of SystemVerilog you write describes either combinational gates or D Flip-Flops. **All blocks in your file exist simultaneously in physical silicon and operate concurrently in parallel.**

---

## 2. Basic Syntax & Modules

### 2.1 The Module Interface
In Digital Logic, you drew functional block diagrams with input wires entering from the left and output wires leaving to the right. In SystemVerilog, that block is called a `module`:

```systemverilog
module traffic_light #(
    parameter int TICKS_PER_SEC = 30_720_000   // Compile-time constant
)(
    input  logic clk,                          // 30.720 MHz oscillator clock
    input  logic reset_n,                      // Active-low reset (0 = RESET, 1 = RUN)
    output logic led_red,                      // Red LED cathode (0 = ON, 1 = OFF)
    output logic led_yel,                      // Yellow LED cathode
    output logic led_grn                       // Green LED cathode
);

    // Internal wires, registers, and FSM logic live here

endmodule
```

### 2.2 Core Types and Constants
* **`logic`**: The standard data type for signals. It represents a physical digital line that can carry a `0`, `1`, or high-impedance state.
* **`parameter`**: A compile-time constant. It uses no physical silicon gates. It allows you to configure things like timer durations. In simulation, you can override `TICKS_PER_SEC = 10` so tests finish in milliseconds instead of waiting 30 million clock cycles.  The simulator is much slower than real time, so we speed it up this way.
* **Buses (Vectors):** To bundle wires together:
  * `logic [2:0] leds;` creates a 3-bit wide bus: `leds[2]`, `leds[1]`, and `leds[0]`.
  * `logic [31:0] timer;` creates a 32-bit counter register.
* **Bit Literals:** Written as `<width>'<base><value>`:
  * `1'b0` = 1-bit binary zero.
  * `3'b101` = 3-bit binary value (decimal 5).
  * `8'hFF` = 8-bit hexadecimal value (decimal 255).
  * `32'd30_720_000` = 32-bit decimal number (underscores are ignored and aid readability).

---

## 3. Combinational Logic: Gates and Muxes

In Digital Logic, you built truth tables, decoders, and multiplexers. In SystemVerilog, you express combinational circuits in two ways:

### Continuous Assignment (`assign`)
Best for simple logic gates and multiplexers:
```systemverilog
// 2-to-1 MUX: output = (sel) ? in_a : in_b;
assign led_red = (current_state == ST_RED) ? 1'b0 : 1'b1;
```

### Combinational Procedural Block (`always_comb`)
Best for complex decision logic, state decoders, and priority trees:
```systemverilog
always_comb begin
    // 1. DEFAULT ASSIGNMENT: Prevents unintended latches!
    next_state = current_state; 

    // 2. Decoder / Selection
    case (current_state)
        ST_RED: begin
            if (timer_expired)
                next_state = ST_RED_YEL;
        end
        ST_RED_YEL: begin
            if (timer_expired)
                next_state = ST_GRN;
        end
        default: next_state = ST_RED;
    endcase
end
```

> ⚠️ **The Golden Rule of Combinational Logic:**  
> A combinational circuit must have a defined output for **every possible combination** of inputs. If you write an `if` without an `else`, or a `case` without covering all cases, the synthesis tool must invent memory to "remember" the old value when that condition is not met. It does this by creating a **transparent latch**.  
> **Unintended latches break timing and are strictly forbidden.** Always provide a default value at the top of an `always_comb` block or include a `default:` branch in your `case` statements.

---

## 4. Sequential Logic: Registers and Clocks

In Digital Logic, you learned that a D Flip-Flop captures the voltage present at its $D$ input right at the rising edge of the clock signal and holds it at $Q$.

```
         D ----+             +---- Q
               |  D-Type FF  |
       clk --->|>            |
               +-------------+
```

In SystemVerilog, a bank of D Flip-Flops is described using an `always_ff` block:

```systemverilog
always_ff @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        // Asynchronous reset: triggers immediately when reset_n drops to 0
        current_state <= ST_RED;
        counter       <= '0;          // '0 clears all bits of the bus
    end else begin
        // Synchronous clock: triggers ONLY on the rising edge of clk
        current_state <= next_state;
        counter       <= counter + 1;
    end
end
```

### The Two Assignment Operators (Crucial Distinction)

SystemVerilog has two different assignment operators. Using the wrong one will cause simulation mismatches and hardware bugs:

| Operator | Name | Where to use | Hardware Meaning |
| :--- | :--- | :--- | :--- |
| `=` | **Blocking** | Inside `always_comb` | Immediate wire connection. Evaluates sequentially like software. |
| `<=` | **Non-blocking** | Inside `always_ff` | **D Flip-Flop transfer.** All assignments update simultaneously on the clock edge. |

#### Why `<=` Matters for Hardware Registers:
Think of a 2-stage shift register (passing value $A \rightarrow B \rightarrow C$ on each clock edge):
```systemverilog
// CORRECT HARDWARE (Non-blocking):
// Both flip-flops read their old inputs simultaneously at the clock edge.
always_ff @(posedge clk) begin
    reg_b <= reg_a;
    reg_c <= reg_b; // reg_c gets the value reg_b had BEFORE this clock tick!
end

// BROKEN (Blocking):
// Behaves like serial software. reg_b updates immediately, and reg_c catches the new value!
always_ff @(posedge clk) begin
    reg_b = reg_a;
    reg_c = reg_b;  // reg_c gets the updated reg_a value in the SAME cycle!
end
```

**Golden Rules:**
* Inside `always_comb` $\rightarrow$ use `=`
* Inside `always_ff` $\rightarrow$ use `<=`
* Never mix them in the same block.

---

## 5. Building Finite State Machines (FSMs)

In Digital Logic, you learned the canonical Moore Machine architecture:
1. **State Register:** D Flip-Flops storing the current state vector.
2. **Next-State Logic:** Combinational logic computing $S_{\text{next}} = f(S_{\text{current}}, \text{Inputs})$.
3. **Output Logic:** Combinational logic computing $\text{Outputs} = g(S_{\text{current}})$.

```
                 +-------------------+
      Inputs --->|                   |
                 |  Next-State Logic |--+
           +---->|  (always_comb)    |  |
           |     +-------------------+  |
           |                            | S_next
           |       +---------------+    |
           |       | State Register|    |
           +-------|  (always_ff)  |<---+
   S_current       | [D Flip-Flops]|
           |       +---------------+
           |               ^
           v               | clk
     +---------------+
     |  Output Logic |
     | (always_comb) |---> Outputs (LEDs)
     +---------------+
```

The industry standard structure is the **multi-block FSM pattern**:

### Step 1: Enumerate States with Human-Readable Names
Avoid raw binary numbers like `2'b00` or `2'b01`. Define an enumerated type:
```systemverilog
typedef enum logic [1:0] {
    ST_RED     = 2'b00,
    ST_RED_YEL = 2'b01,
    ST_GRN     = 2'b10,
    ST_YEL     = 2'b11
} state_t;

state_t current_state, next_state;
```

### Step 2: The State Register (`always_ff`)
This synthesizes the flip-flops for the state:
```systemverilog
always_ff @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        current_state <= ST_RED;
    end else begin
        current_state <= next_state;
    end
end
```

### Step 3: Next-State Logic (`always_comb`)
Combinational logic determining which state comes next based on timers or inputs:
```systemverilog
always_comb begin
    next_state = current_state; // Default: hold state

    case (current_state)
        ST_RED: begin
            if (timer_done) next_state = ST_RED_YEL;
        end
        ST_RED_YEL: begin
            if (timer_done) next_state = ST_GRN;
        end
        ST_GRN: begin
            if (timer_done) next_state = ST_YEL;
        end
        ST_YEL: begin
            if (timer_done) next_state = ST_RED;
        end
        default: next_state = ST_RED;
    endcase
end
```

### Step 4: Output Logic (`always_comb`)
In a Moore machine, outputs depend **only on the current state**:
```systemverilog
// HARDWARE NOTE: Open-Drain Pins
// 1'b0 connects cathode to GND (LED ON)
// 1'b1 disconnects cathode (LED OFF)
always_comb begin
    // Safe default: All LEDs OFF
    led_red = 1'b1;
    led_yel = 1'b1;
    led_grn = 1'b1;

    case (current_state)
        ST_RED: begin
            led_red = 1'b0; // Red ON
        end
        ST_RED_YEL: begin
            led_red = 1'b0; // Red ON
            led_yel = 1'b0; // Yellow ON
        end
        ST_GRN: begin
            led_grn = 1'b0; // Green ON
        end
        ST_YEL: begin
            led_yel = 1'b0; // Yellow ON
        end
        default: begin
            // Maintain all OFF
        end
    endcase
end
```

---

## 6. Real-Time Hardware Timers on the FPGA

The Pico Dev-iCE board has a dedicated hardware crystal oscillator running at **30.720 MHz**. This means `clk` toggles 30,720,000 times every second.

### Never Generate a "Slow Clock" Wire
In introductory classes, students sometimes try to slow things down by dividing the clock signal directly:
```systemverilog
// NEVER DO THIS ON AN FPGA:
always_ff @(posedge clk) slow_clk <= ~slow_clk;
always_ff @(posedge slow_clk) ... // DANGEROUS!
```
In real silicon, clocks must travel over dedicated, low-skew physical clock networks. Creating a clock from logic gates introduces clock skew, jitter, and hold-time violations.

### The Correct Way: Synchronous Enable Counters
Keep everything clocked by the 30.720 MHz master clock. Use a counter register to track elapsed cycles:

```systemverilog
// To store 30,720,000 we need at least 25 bits (2^25 = 33,554,432).
// A 32-bit register provides plenty of room:
logic [31:0] tick_counter;
logic        one_sec_pulse;

always_ff @(posedge clk or negedge reset_n) begin
    if (!reset_n) begin
        tick_counter  <= 0;
        one_sec_pulse <= 1'b0;
    end else begin
        // Beware the Off-by-One error!
        // To count N events starting at index 0, count from 0 to N-1.
        if (tick_counter >= (TICKS_PER_SEC - 1)) begin
            tick_counter  <= 0;
            one_sec_pulse <= 1'b1; // High for exactly ONE clock tick
        end else begin
            tick_counter  <= tick_counter + 1;
            one_sec_pulse <= 1'b0;
        end
    end
end
```

When `one_sec_pulse` goes high, you can increment a state-duration counter or trigger your FSM transitions.

---

## 7. Writing a Self-Checking Testbench

A testbench is a software simulation harness. It is never synthesized into silicon; instead, it provides virtual inputs and checks the module's outputs against expectations.

### Anatomy of a Testbench
1. **No Ports:** The module header is empty (`module traffic_tb;`).
2. **Device Under Test (DUT):** An instance of your hardware module.
3. **Clock Generator:** An `always` loop that periodically flips the clock bit.
4. **Stimulus & Checks:** An `initial` block that applies reset, waits for clock edges, and verifies outputs using `if` statements and `$fatal` or `$display`.

### Compiler Directives & `` `timescale ``

Lines starting with a backtick (`` ` ``) are **compiler directives**—instructions processed by the simulator/compiler before parsing your hardware code (analogous to `#include` or `#define` in C).

```systemverilog
`timescale 1ns / 1ps
```

* **Format:** `` `timescale <time_unit> / <time_precision> ``
* **Time Unit (`1ns`):** Sets the measurement unit for simulation delays. When you write `#5`, the simulator waits 5 units—meaning **5 ns**.
* **Precision (`1ps`):** Sets the simulator's internal rounding accuracy. Time steps and transitions are calculated and rounded to the nearest **1 ps** (0.001 ns).

```systemverilog
`timescale 1ns / 1ps

module traffic_tb;

    // 1. Signals to wire up to the DUT
    logic clk;
    logic reset_n;
    logic led_red;
    logic led_yel;
    logic led_grn;

    // 2. Instantiate the DUT with overridden parameters
    // In real hardware: TICKS_PER_SEC = 30_720_000
    // In simulation:    TICKS_PER_SEC = 10 (10 ticks = 1 second)
    traffic_light #(
        .TICKS_PER_SEC(10)
    ) dut (
        .clk     (clk),
        .reset_n (reset_n),
        .led_red (led_red),
        .led_yel (led_yel),
        .led_grn (led_grn)
    );

    // 3. Generate a Clock: 10ns period (100 MHz simulation clock)
    initial clk = 0;
    always #5 clk = ~clk; // Toggles every 5 time units

    // 4. Test Sequence & Automated Verification
    initial begin
        $display("=== STARTING TRAFFIC LIGHT TEST ===");

        // --- Step A: Test Reset State ---
        reset_n = 1'b0;
        #20; // Hold reset low for 20 ns (2 clock cycles)

        // At reset: Red LED must be ON (0), Yellow & Green OFF (1)
        if (led_red !== 1'b0 || led_yel !== 1'b1 || led_grn !== 1'b1) begin
            $fatal(1, "[FAIL] Reset failed! Got R=%b, Y=%b, G=%b (Expected 0, 1, 1)", 
                   led_red, led_yel, led_grn);
        end
        $display("[PASS] Reset state verified: Red LED is active.");

        // --- Step B: Release Reset ---
        @(negedge clk);
        reset_n = 1'b1;

        // --- Step C: Verify Red Duration (e.g., 3 seconds = 30 clock ticks) ---
        repeat (30) @(posedge clk);

        // Allow 1ns for combinational logic propagation after the clock edge
        #1;
        if (led_red !== 1'b0 || led_yel !== 1'b0 || led_grn !== 1'b1) begin
            $fatal(1, "[FAIL] Did not enter RED+YELLOW after 3 seconds! Got R=%b, Y=%b, G=%b", 
                   led_red, led_yel, led_grn);
        end
        $display("[PASS] RED -> RED+YELLOW transition verified on time.");

        // (Continue checking your other state transitions here...)

        $display("=== ALL VERIFICATION CHECKS PASSED! ===");
        $finish; // End simulation cleanly
    end

endmodule
```

### Essential Testbench Keywords:
* `#5`: Delay simulation time by 5 units.
* `@(posedge clk)`: Wait until the next rising clock edge.
* `repeat (N) @(posedge clk)`: Wait for $N$ consecutive clock edges.
* `$display(...)`: Prints formatted messages to the console (like `printf`).
* `$fatal(1, ...)`: Prints an error and terminates simulation with an exit failure code.
* `!==` and `===`: Strict equality operators that check four-state logic (`0`, `1`, `X` unknown, `Z` disconnected).

---

## 8. EDA Toolchain: Simulation to Bitstream

Here is how code moves through the OSS CAD Suite tools to become hardware:

```
               [ SystemVerilog Files: .sv ]
                     /                \
        (Simulation)/                  \(Synthesis)
                   v                    v
          [ iverilog / vvp ]          [ yosys ]
                   |                    | (Synthesized Netlist: .json)
              Verification              v
              (PASS/FAIL)         [ nextpnr-ice40 ]
                                        | (Routed Physical Design: .asc)
                                        v
                                    [ icepack ]
                                        |
                               (traffic.bin)
                                        |
                                        v
                             [ Pico Dev-iCE CRAM ]
```

### Command 1: Simulation (Icarus Verilog)
Compile your design and testbench together using the `-g2012` flag (which enables modern SystemVerilog support):
```bash
iverilog -g2012 -o sim.vvp traffic_light.sv traffic_tb.sv
vvp sim.vvp
```
If your testbench passes, it displays your success banner. If an assertion fails, it reports the exact cycle and terminates.

### Command 2: Logic Synthesis (Yosys)
Synthesizes your SystemVerilog into basic iCE40 logic cells:
```bash
yosys -p "read_verilog -sv traffic_light.sv; synth_ice40 -top traffic_light -json traffic.json"
```

### Command 3: Place & Route (NextPNR)
Maps the logic cells to physical Look-Up Tables (LUTs) and routes them to the physical pins defined in `ddc_sdr.pcf`:
```bash
nextpnr-ice40 --up5k --package sg48 --json traffic.json --pcf ddc_sdr.pcf --asc traffic.asc
```

### Command 4: Bitstream Generation (Icepack)
Packs the routed design into an FPGA configuration bitstream:
```bash
icepack traffic.asc traffic.bin
```

---

## 9. AI Audit: Common Hallucinations to Catch

In Parts 2 and 3 of the lab, you will prompt an AI to generate the FSM. While LLMs know SystemVerilog syntax, **they regularly make fundamental digital architecture mistakes**. 

Inspect the generated code for these common bugs:

| Common AI Bug | What the AI Wrote | Why It Fails in Physical Hardware | How to Fix It |
| :--- | :--- | :--- | :--- |
| **Inverted LED Logic** | Set outputs to `1'b1` to turn an LED ON. | The board pins are **Open-Drain** cathodes. Outputting `1'b1` turns the LED **OFF**; `1'b0` turns it **ON**. | Invert the output assignments. |
| **Off-by-One Counter** | `if (counter == TICKS_PER_SEC)` | Counting from $0$ up to $N$ takes $N+1$ clock ticks. State transitions will run slow. | Use `counter >= (TICKS_PER_SEC - 1)`. |
| **Inferred Latches** | Forgot an `else` branch or missing cases inside `always_comb`. | The synthesis tool creates unwanted transparent latches that cause race conditions. | Add default output values at the beginning of the `always_comb` block. |
| **Single Massive Always Block** | Packed state registers, next-state logic, and output decoding into one single `always_ff`. | Creates output register delays and makes verification difficult. | Separate the code into state register (`always_ff`), next-state (`always_comb`), and output logic (`always_comb`). |
| **Blocking in Sequential** | Used `=` instead of `<=` inside an `always_ff` block. | Creates simulation race conditions that don't match silicon behavior. | Ensure all assignments in `always_ff` use `<=`. |